# Creating a Simple Agent with Tracing

In [4]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [5]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [6]:
nutrition_agent = Agent(
    name="nutrition_assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """
)

Let's execute the Agent:

In [7]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="nutrition_assistant", ...)
- Final output (str):
    Bananas are a healthy, convenient source of potassium, vitamin B6, vitamin C, and fiber. A medium banana (~105 kcal) has about 27 g carbs, 3 g fiber, and 14 g sugar.
    
    Benefits:
    - Supports heart and nerve function, digestion, and satiety.
    - Moderates blood sugar when paired with protein or fat.
    
    Notes:
    - Go for ripe or slightly underripe depending on taste; ripe bananas have more sugar but easier to digest.
    - For diabetes or strict carb counting, mind portion size and pair with protein/fat.
    - Wash before eating; choose organic if pesticides are a concern.
    
    Overall: they can be part of a balanced diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [8]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Overall healthy when eaten as part of a balanced diet.

- Nutrients: good source of potassium, vitamin C, vitamin B6, and dietary fiber.
- Carbs/sugar: natural sugars; moderate GI. One medium banana ~105 calories.
- Fiber: supports digestion; unripe bananas have more resistant starch (lower GI).
- Considerations: may be high in sugar for some, and very ripe bananas have higher sugar content. If you have kidney disease or specific medical conditions, consult a clinician about potassium intake.
- Tips: pair with protein or fat (e.g., peanut butter) for staying power; store at room temperature or refrigerate ripe bananas to slow browning.

_Good Job!_